# Inventory RAG Agent: walkthrough

This notebook runs the whole pipeline one step at a time and shows what each step produces:

1. **The sample data**: three PDFs and a SQLite database for a fictional distributor, Acme Distribution
2. **PDF → Markdown** with Docling
3. **Chunking** along section headings, with tables kept whole
4. **Embedding and indexing** in Chroma
5. **The three tools**, called directly
6. **The agent** answering questions that need documents, data and arithmetic together, including a follow-up that relies on chat history

```
 PDFs ──Docling──► Markdown ──chunk──► Chroma ──── search_documents ──┐
                                                                      │
 SQLite (stock, sales, purchase orders) ────────── run_sql ───────────┼──► agent (gpt-oss-120b on Groq) ──► answer
                                                                      │
 policy formula ─────────────────────────── calculate_reorder_point ──┘
```

The code lives in `ingest.py` and `agent.py`, and this notebook imports it, so the notebook and the scripts always run the same code. The chat interface is `app.py`.

In [1]:
import os
import sqlite3

import pandas as pd

import agent    # loads GROQ_API_KEY from the .env file at the repository root
import ingest

print("Groq API key found:", bool(os.environ.get("GROQ_API_KEY")))

Groq API key found: True


## 1. The sample data

Acme Distribution is fictional, and so is all of its data.

- **`inventory_policy.pdf`**: ABC classes and their service levels, the safety-stock and reorder-point formulas, review rules and exceptions
- **`supplier_agreements.pdf`**: lead times, minimum order quantities, payment terms and on-time rates for four suppliers, plus penalties
- **`demand_warehouse_report.pdf`**: the Q4 demand forecast, a November promotion, warehouse capacity and key risks

The database holds the live numbers: products, stock on hand and on order, 13 weeks of sales, and open purchase orders.

In [2]:
print("PDFs:", [p.name for p in sorted(ingest.PDF_DIR.glob("*.pdf"))])

db = sqlite3.connect(f"{agent.DB_PATH.as_uri()}?mode=ro", uri=True)
for (table,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table'"):
    print(f"{table}: {db.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]} rows")

stock = pd.read_sql("""SELECT i.sku, p.name, p.abc_class, i.on_hand, i.on_order, i.reorder_point, s.name AS supplier
                        FROM inventory i JOIN products p USING (sku) JOIN suppliers s USING (supplier_id)""", db)
db.close()
stock

PDFs: ['demand_warehouse_report.pdf', 'inventory_policy.pdf', 'supplier_agreements.pdf']
suppliers: 4 rows
products: 8 rows
inventory: 8 rows
sales_history: 104 rows
purchase_orders: 4 rows


,sku,name,abc_class,on_hand,on_order,reorder_point,supplier
0,EL-1001,Wireless Sensor,A,240,0,900,Delta Electronics
1,EL-1002,Control Board,A,310,200,500,Delta Electronics
2,FS-2001,M6 Steel Bolt (box),B,2100,0,1200,Pacific Fasteners
3,FS-2002,Hex Nut (box),C,900,500,700,Pacific Fasteners
4,PK-3001,Shipping Box Large,A,1500,0,2200,Summit Packaging
5,PK-3002,Bubble Wrap Roll,B,260,300,300,Summit Packaging
6,CP-4001,Linear Actuator,B,85,0,120,Nova Components
7,CP-4002,Rubber Gasket,C,4000,0,1500,Nova Components


## 2. PDF → Markdown with Docling

Plain PDF text extraction turns tables into a jumble of numbers. Docling runs layout analysis and table-structure recognition, so tables come out as real Markdown tables and headings as Markdown headings. Both matter for the next step. The Markdown is saved to `data/markdown/`.

The first run downloads Docling's layout models, which takes a minute or two.

In [3]:
for name, size in ingest.convert_pdfs().items():
    print(f"{name} -> {size:,} characters")

print("\n--- supplier_agreements.md (start) ---\n")
print((ingest.MD_DIR / "supplier_agreements.md").read_text(encoding="utf-8")[:1200])

demand_warehouse_report.pdf -> 1,692 characters
inventory_policy.pdf -> 1,977 characters
supplier_agreements.pdf -> 1,665 characters

--- supplier_agreements.md (start) ---

## Supplier Agreements Summary FY2026

## 1. Overview

This document summarises commercial terms with our four active suppliers. Lead times are quoted from purchase-order confirmation to goods receipt at our dock.

## 2. Supplier Terms

| Supplier          |   Lead Time (days) |   MOQ (units) | Payment Terms   | On-Time Delivery   |
|-------------------|--------------------|---------------|-----------------|--------------------|
| Nova Components   |                 14 |           200 | Net 45          | 96%                |
| Pacific Fasteners |                  7 |           500 | Net 30          | 91%                |
| Delta Electronics |                 28 |           100 | Net 60          | 84%                |
| Summit Packaging  |                 10 |           300 | Net 30          | 98%                |



## 3. Chunking

The documents are short and well structured, so the chunker follows their structure instead of cutting at a fixed size:

- **Split at every heading**, so a chunk never mixes two sections.
- **Never split a table.** Each table becomes one chunk; half a table is useless.
- **Pack prose** paragraphs into chunks of up to about 800 characters.
- **Prefix every chunk with `[document > section]`**, so a chunk still makes sense when it is retrieved on its own, and the agent can cite where it came from.

In [4]:
chunks = ingest.load_chunks()
print(pd.DataFrame([c["metadata"] for c in chunks]).groupby(["source", "type"]).size().unstack(fill_value=0))

table_chunk = next(c for c in chunks if c["metadata"]["type"] == "table")
text_chunk = next(c for c in chunks if c["metadata"]["type"] == "text")
print("\n--- a table chunk ---\n" + table_chunk["text"])
print("\n--- a text chunk ---\n" + text_chunk["text"])

type                     table  text
source                              
demand_warehouse_report      2     3
inventory_policy             2     5
supplier_agreements          2     3

--- a table chunk ---
[demand_warehouse_report > 2. Demand Forecast by Category]
| Category    |   Q3 Actual Units |   Q4 Forecast Units | Change   |
|-------------|-------------------|---------------------|----------|
| Electronics |             4,200 |               5,300 | +26%     |
| Fasteners   |             9,800 |              10,600 | +8%      |
| Packaging   |             6,100 |               7,500 | +23%     |
| Components  |             3,500 |               3,800 | +9%      |

--- a text chunk ---
[demand_warehouse_report > 1. Executive Summary]
Demand is expected to rise 18% in Q4 versus Q3 driven by the holiday season and a planned promotion on Wireless Sensors and Shipping Boxes in November. Planners should build stock ahead of the promotion, especially for items with long supplier lead

## 4. Embed and index in Chroma

`all-MiniLM-L6-v2` is a small, free sentence-transformers model that runs locally (384 dimensions). Chroma embeds each chunk with it and stores the collection in `data/chroma_db/`, using cosine distance (0 = same direction, lower is closer). Rebuilding deletes the old collection first, so running this twice never duplicates chunks.

A quick check: does a question about a supplier find the right sections?

In [5]:
collection = ingest.build_index(chunks)
print("Indexed:", collection.count(), "chunks")

hits = collection.query(query_texts=["How long does Delta Electronics take to deliver?"], n_results=3)
for doc, dist in zip(hits["documents"][0], hits["distances"][0]):
    print(f"\n(distance {dist:.3f})\n{doc[:350]}")

Indexed: 17 chunks

(distance 0.276)
[demand_warehouse_report > 4. Key Risks]
1) Delta Electronics' 28-day lead time and 84% on-time rate put electronics promotion stock at risk. 2) Port congestion may add 5 to 7 days to inbound ocean freight. 3) Promotion overlap may exhaust capacity in the Bulk Storage zone.

(distance 0.348)
[supplier_agreements > 4. Penalties and Reliability]
Suppliers with on-time delivery below 90% for two consecutive quarters enter a performance review. Delta Electronics is currently on review; planners should add a 20% lead-time buffer when ordering from them.

(distance 0.433)
[supplier_agreements > 3. Expedite Rules]
Nova Components and Summit Packaging offer expedited shipping at a 15% price premium, cutting lead time by 50%. Pacific Fasteners can expedite to 3 days for a flat USD 400 fee. Delta Electronics does not offer expedited service because of ocean freight.


## 5. The tools

The model is given three tools, each described by a name, a short description and a JSON Schema of its arguments:

| Tool | What it does |
|---|---|
| `search_documents(query, n_results=4)` | Semantic search over the indexed PDFs; returns each chunk with its document and section |
| `run_sql(query)` | Runs one `SELECT` (or `WITH`) query on the database and returns up to 100 rows |
| `calculate_reorder_point(avg_daily_demand, demand_std_daily, lead_time_days, service_level)` | Safety stock `SS = z · σ_d · √L` and reorder point `ROP = d · L + SS` |

Called directly, without the model, they show exactly what the model gets back. `run_sql` has two layers of protection: it refuses anything that does not start with `SELECT` or `WITH`, and it opens the database read-only, so even a write hidden inside a `WITH` query fails. SQLite has no standard-deviation function, so the tool adds `STDEV()`, which the demand-variability questions need.

In [6]:
print(agent.run_sql("SELECT name, lead_time_days, moq, on_time_pct FROM suppliers"))
print(agent.run_sql("DELETE FROM inventory"))
print(agent.run_sql("WITH x AS (SELECT 1) DELETE FROM inventory"))
print(agent.run_sql("SELECT sku, ROUND(AVG(units_sold) / 7, 1) AS daily, ROUND(STDEV(units_sold) / SQRT(7), 1) AS sd_daily "
                    "FROM sales_history WHERE sku = 'EL-1001'"))
print(agent.calculate_reorder_point(avg_daily_demand=30, demand_std_daily=5, lead_time_days=28,  # example inputs
                                    service_level=0.99))

{'columns': ['name', 'lead_time_days', 'moq', 'on_time_pct'], 'rows': [('Nova Components', 14, 200, 96.0), ('Pacific Fasteners', 7, 500, 91.0), ('Delta Electronics', 28, 100, 84.0), ('Summit Packaging', 10, 300, 98.0)]}
{'error': 'Only SELECT queries are allowed.'}
{'error': 'attempt to write a readonly database'}
{'columns': ['sku', 'daily', 'sd_daily'], 'rows': [('EL-1001', 33.7, 11.6)]}
{'z': 2.33, 'safety_stock': 61.6, 'reorder_point': 901.6}


## 6. The agent

`InventoryAgent.ask()` runs the tool-calling loop:

1. Send the system prompt, any chat history and the question to `openai/gpt-oss-120b` on Groq, together with the three tool descriptions.
2. If the model requests tools, run each one and send the result back as a tool message.
3. Repeat until the model answers without requesting a tool, for at most 10 steps.

The system prompt includes the database schema (so the model can write SQL without exploring), tells it to look facts up instead of guessing, to show its numbers and formula, and to cite documents. It also fixes today's date at 2026-10-16, the day the sample data was taken. Two of its rules were added after a first run went wrong (see *What the runs show* at the end): how to turn weekly sales into a daily standard deviation, and to check the supplier agreements before using a lead time.

Four safeguards handle problems seen in practice:
- gpt-oss sometimes writes a formula such as `30.7/(7**0.5)` into a tool call's JSON, and Groq rejects the whole call (`tool_use_failed`). The agent tells the model what happened and lets it try again.
- If the model is still calling tools after 10 steps, the agent asks for an answer from the results gathered so far. It does this in a fresh conversation with the results written out as text: once a conversation contains tool calls, gpt-oss tries to call another tool even when `tool_choice="none"` or no tools are offered, and Groq rejects the request.
- gpt-oss's hidden reasoning counts toward the token limit, so a long answer can be cut off (`finish_reason="length"`). The system prompt asks for answers under 300 words, and if one is still cut off, the agent asks again from the gathered results with low reasoning effort.
- A tool that fails (bad SQL, a wrong argument) returns its error to the model as the tool result, so the model can correct itself.

Each question below prints the tool calls as they happen, then the answer. Answers vary a little between runs; the ones shown are from the saved run.

In [7]:
inventory_agent = agent.InventoryAgent()


def ask(question, history=None):
    print(f"Q: {question}\n")
    answer = inventory_agent.ask(question, history,
                                 on_tool=lambda name, args: print(f"  tool: {name} {args}"))
    print(f"\nA: {answer}")
    return answer


answer_1 = ask("Which SKUs are below their reorder point once on-order stock is counted?")

Q: Which SKUs are below their reorder point once on-order stock is counted?



  tool: run_sql {'query': 'SELECT sku, on_hand, on_order, reorder_point, (on_hand + on_order) AS total_stock FROM inventory;'}



A: **SKUs currently below their reorder point (including on‑order stock):**

| SKU      | On‑hand | On‑order | Reorder point | Total stock (hand + order) |
|----------|---------|----------|---------------|----------------------------|
| EL-1001  | 240     | 0        | 900           | 240 |
| PK-3001  | 1 500   | 0        | 2 200         | 1 500 |
| CP-4001  | 85      | 0        | 120           | 85 |

These values come directly from the **inventory** table (run_sql query). All three SKUs have total stock (on‑hand + on‑order) less than their defined reorder point, indicating they should be considered for replenishment.


This one needs only the database. The next question needs all three tools: the service level for the SKU's ABC class is in the policy PDF, the lead time and the demand history are in the database, and the formula is the calculator's job.

In [8]:
question_2 = "What is the safety stock for EL-1001 at the service level required by its ABC class?"
answer_2 = ask(question_2)

Q: What is the safety stock for EL-1001 at the service level required by its ABC class?



  tool: run_sql {'query': "SELECT sku, abc_class, supplier_id FROM products WHERE sku = 'EL-1001';"}


  tool: search_documents {'n_results': 5, 'query': 'ABC class service level policy'}


  tool: run_sql {'query': 'SELECT supplier_id, lead_time_days FROM suppliers WHERE supplier_id = 3;'}


  tool: run_sql {'query': "SELECT week_start, units_sold FROM sales_history WHERE sku = 'EL-1001' ORDER BY week_start;"}


  tool: search_documents {'n_results': 5, 'query': 'supplier 3 lead time buffer agreement'}



A: **Safety Stock for EL‑1001**

1. **ABC class & service level** – EL‑1001 is an **A‑class** SKU (product table).  
   *Policy*: A‑class targets a **99 % service level** (inventory_policy, “ABC Classification”).  
   *Z‑factor*: 2.33 for 99 % (inventory_policy, “Safety Stock and Reorder Point”).

2. **Supplier lead‑time** – Supplier 3 (Delta Electronics) has a quoted lead time of **28 days** (suppliers table).  
   *Agreement buffer*: Delta is under performance review, so a **20 % buffer** is added (supplier_agreements, “Penalties and Reliability”).  
   Effective lead time = 28 × 1.20 = **33.6 days** → use **34 days** for the calculation.

3. **Demand statistics** – Weekly sales for the last 13 weeks (sales_history) give:

   * Total units = 3 070 → average weekly demand = 3 070 / 13 ≈ 236.15 units  
   * Average daily demand = 236.15 / 7 ≈ **33.74 units/day**  

   * Weekly standard deviation ≈ 30.74 units (sample) → daily standard deviation  
     σ₍d₎ = 30.74 / √7 ≈ **11.62 units

### Short-term memory

The chat interface sends the last 10 messages with each new question. Here the same happens by hand: the follow-up question says "it", which only makes sense with the previous turn as history.

In [9]:
history = [{"role": "user", "content": question_2}, {"role": "assistant", "content": answer_2}]
answer_3 = ask("How would that change at a 95% service level instead?", history)

Q: How would that change at a 95% service level instead?



  tool: calculate_reorder_point {'avg_daily_demand': 33.74, 'demand_std_daily': 11.62, 'lead_time_days': 34, 'service_level': 0.95}



A: **Safety‑stock at a 95 % service level**

The only factor that changes is the Z‑factor (the number of standard deviations that corresponds to the desired service level).

| Parameter | Value |
|-----------|-------|
| Average daily demand (d) | 33.74 units/day |
| Daily demand standard deviation (σd) | 11.62 units/day |
| Effective lead time (L) | 34 days (28 days × 1.20 buffer) |
| Z‑factor for 95 % service level | **1.65** (inventory policy) |

**Formula**

```
Safety Stock = Z × σd × √L
```

**Calculation**

```
√L = √34 ≈ 5.83
Safety Stock = 1.65 × 11.62 × 5.83 ≈ 111.8 ≈ **112 units**
```

So, moving from the A‑class 99 % target (Z = 2.33, safety stock ≈ 157 units) down to a 95 % service level reduces the safety stock to **about 112 units** – a reduction of roughly **45 units** (≈ 29 %).  

(If you also need the new reorder point, it would be d × L + SS = 33.74 × 34 + 112 ≈ 1,259 units.)


The last question is about risk, so it relies mainly on the documents: the supplier terms, the promotion in the demand report and the policy's rules.

In [10]:
answer_4 = ask("Given the November promotion, which suppliers are risky and what can we do about it?")

Q: Given the November promotion, which suppliers are risky and what can we do about it?



  tool: search_documents {'n_results': 5, 'query': 'November promotion'}


  tool: run_sql {'query': "SELECT sku, name, supplier_id FROM products WHERE name LIKE '%Wireless%' OR name LIKE '%Sensor%'"}


  tool: run_sql {'query': "SELECT sku, name, supplier_id FROM products WHERE name LIKE '%Box%'"}


  tool: run_sql {'query': "SELECT week_start, units_sold FROM sales_history WHERE sku='EL-1001' ORDER BY week_start"}


  tool: run_sql {'query': "SELECT on_hand, on_order, reorder_point FROM inventory WHERE sku='EL-1001'"}


... (2 outputs left out)

  tool: run_sql {'query': "SELECT sku, on_hand, on_order, reorder_point FROM inventory WHERE sku='PK-3001'"}


  tool: run_sql {'query': "SELECT week_start, units_sold FROM sales_history WHERE sku='PK-3001' ORDER BY week_start"}


  tool: run_sql {'query': "SELECT STDDEV(units_sold) FROM sales_history WHERE sku='PK-3001'"}



A: **Suppliers that are risky for the November promotion**

| SKU (promo) | Supplier | Lead‑time (days) | On‑time delivery | Current on‑hand | Reorder‑point* |
|-------------|----------|------------------|------------------|-----------------|----------------|
| EL‑1001 (Wireless Sensor) | **Delta Electronics** | 28 | 84 % | 240 | 900 (system) → 1 046 (calc) |
| PK‑3001 (Shipping Box Large) | **Summit Packaging** | 10 | 98 % | 1 500 | 2 200 |

*Reorder‑point shown is the system value; a service‑level calculation (95 % service, z = 1.65) gave safety stock ≈ 101.5 and **reorder‑point = 1 046** (calc from `avg_daily_demand * lead_time + safety_stock`).

**Why they are risky**

* **Delta Electronics** – highlighted in *demand_warehouse_report* “Key Risks” as the biggest risk (28‑day lead time, only 84 % on‑time) for the electronics promotion (source: demand_warehouse_report, §4). With only 240 units on hand and a required reorder point of ~1 046, we would be short by ~ 800 units if the pro

## 7. The chat interface

`python app.py` starts a Gradio chat at http://127.0.0.1:7860 with the same agent. Each answer ends with a collapsible **Tools used** list showing every tool call, so you can check how the agent got its numbers. The last 10 messages are passed as history, with the tool lists removed, so follow-up questions like the one above work in the chat too.

Gradio 6 passes chat history as messages whose content is a list of blocks such as `{"type": "text", "text": ...}`, not a plain string. `app.py` reads both forms. Code that keeps only string content would silently drop the whole history, and the chat would lose its memory without any error.

## What the runs show

The answers above are from this notebook's saved run, checked by hand against the documents and the database. The tool results are the same on every run; the model's answers vary.

**Narrow questions with one right answer worked.**
- The SKUs below their reorder point are EL-1001, PK-3001 and CP-4001, the same three a direct query finds.
- The safety stock for EL-1001 is 157 units: class A means a 99% target and z = 2.33 (policy), the daily standard deviation is 30.74 / √7 = 11.62 (sales history), and the lead time is 28 days plus Delta Electronics' 20% review buffer, 33.6 days (supplier agreements). So 2.33 × 11.62 × √33.6 ≈ 157.
- The follow-up reused those inputs from the chat history and changed only z, giving 112 units (it rounded the lead time to 34 days; with 33.6 days it is 111).

**A first run got the safety stock confidently wrong.** It answered 54 units, with the right formula, the right z-factor and a tidy table of sources. Two inputs were wrong:
- It turned the weekly standard deviation into a daily one by dividing by 7, as you would an average. Variability grows with the square root of time, so the right divisor is √7.
- It used Delta's quoted 28-day lead time and never read the supplier agreement's rule to add a 20% buffer for a supplier on review.

Both rules are now in the system prompt. Nothing in the first answer looked wrong; only checking the inputs caught it.

**The broad question failed twice before it worked.** "Given the November promotion..." makes the agent chain about ten tool calls. In the first run it hit the step limit and the forced final answer was rejected, because gpt-oss tried to call yet another tool. In the second run, its long answer was cut off by the token limit. With the flattened fallback and a word limit, this run answers.

**A fluent answer to a broad question still needs checking.** This answer names the right risks with their sources: Delta Electronics' 28-day lead time and 84% on-time rate (demand report, Key Risks), port congestion, and the 2% credit for deliveries more than 5 days late (supplier agreements). But it also suggests air freight for Delta, which the supplier agreements rule out ("Delta Electronics does not offer expedited service"). And it sizes the sensor reorder point with a 95% service level and no buffer, although two questions earlier the same agent used 99% and the buffer for the same SKU. Broad planning questions gather more facts than the model then uses consistently, so they are the place to check the tool trace.